In [5]:
import pandas as pd
df = pd.read_csv("outputs/grid_summary_1777102188.csv", sep=";")

In [6]:
# ── Labels ────────────────────────────────────────────────────────────────────
REBAL_LABELS = {5: "weekly", 3: "3 days", 2: "2 days", 1: "daily"}
REBAL_ORDER  = {5: 0, 3: 1, 2: 2, 1: 3}
MATURITY_LABELS = {0.0833333: "1M", 0.25: "3M"}

df["rebal_label"]    = df["run.rebalancing"].map(REBAL_LABELS)
df["rebal_order"]    = df["run.rebalancing"].map(REBAL_ORDER)
df["maturity_label"] = df["run.maturity"].map(MATURITY_LABELS).fillna(df["run.maturity"].astype(str))

# ── Colonnes MultiIndex identiques au papier ──────────────────────────────────
COLS = pd.MultiIndex.from_tuples([
    ("Delta Hedging",      "Mean Cost"),
    ("Delta Hedging",      "S.D. Cost"),
    ("RL Optimal Hedging", "Mean Cost"),
    ("RL Optimal Hedging", "S.D. Cost"),
    ("Y(0)",               "improvement"),
])

def pct(x):
    return f"{x:.1f}%"

def build_table(sub: pd.DataFrame) -> pd.DataFrame:
    sub = sub.sort_values("rebal_order")
    rows = {}
    for _, row in sub.iterrows():
        rows[row["rebal_label"]] = [
            pct(row["mean_bm"]),
            pct(row["std_bm"]),
            pct(row["mean_rl"]),
            pct(row["std_rl"]),
            pct(row["improvement_pct"]),
        ]
    result = pd.DataFrame.from_dict(rows, orient="index", columns=COLS)
    result.index.name = "Rebal Freq"
    return result

# ── Construction du dictionnaire de tableaux ──────────────────────────────────
tables = {}

# Clé de groupement : agent + process + maturity (+ actor_objective si QRDDPG)
for (agent, process, maturity, objective), group in df.groupby(
    ["agent", "process", "maturity_label", "hedging.agent.actor_objective"]
):
    if agent == "QRDDPG":
        key = f"{agent}_{process}_{objective}_{maturity}"
    else:
        key = f"{agent}_{process}_{maturity}"
    tables[key] = build_table(group)

# ── Aperçu ────────────────────────────────────────────────────────────────────
for name, tbl in tables.items():
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print('─'*60)
    print(tbl.to_string())


────────────────────────────────────────────────────────────
  DeepDPG_GBM_1M
────────────────────────────────────────────────────────────
           Delta Hedging           RL Optimal Hedging                  Y(0)
               Mean Cost S.D. Cost          Mean Cost S.D. Cost improvement
Rebal Freq                                                                 
weekly             60.6%     66.9%              57.0%     68.0%        1.2%
3 days             72.2%     54.5%              63.2%     54.6%        5.7%
2 days             79.9%     47.1%              66.3%     50.6%        5.6%
daily             103.7%     39.7%              87.5%     40.3%        9.3%

────────────────────────────────────────────────────────────
  DeepDPG_GBM_3M
────────────────────────────────────────────────────────────
           Delta Hedging           RL Optimal Hedging                  Y(0)
               Mean Cost S.D. Cost          Mean Cost S.D. Cost improvement
Rebal Freq                          

In [10]:
from better_tables import build_table

In [ ]:
for name, tbl in tables.items():
    print(name)
    table_latex = build_table(tbl).to_latex(
        save_as_pdf=True, save_as_tex=True, path=f"outputs/tables/{name}.pdf")
